In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch
import copy
from torch.utils.data import DataLoader
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from tqdm import tqdm

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import SimpleTrainer, SITrainer, IntervalTrainer
from src.data_utils import (
    get_mnist_tasks,
    _extract_targets,
    get_context_sets,
    create_holdout_set,
)
from src.utils import InContextHead
from src import models
from src.regulariser import MultiRegulariser, L2Regulariser, UnbiasRegulariser

### Class Incremental Learning

In [4]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [ ]:
model = models.get_mnist_model(device="cuda")

trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=1, batch_size=300)
trainer.test(test_tasks[0:2])

Training Epochs: 100%|██████████████████████████████████████████| 1/1 [00:06<00:00,  6.03s/it, val_loss=0.399, val_acc=0.93]


Test Results: [(0.3914, 0.9332), (9.5302, 0.0)]


[(0.3914, 0.9332), (9.5302, 0.0)]

In [ ]:
si_trainer = SITrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.7,
    primal_learning_rate=0.5,
    paradigm="CIL",
)

loader = DataLoader(
    train_tasks[1],
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
samples = next(iter(loader))
# Compute bounds for task 0
si_trainer.compute_rashomon_set(test_tasks[0], si_batch=samples, prune_prop=0.8)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    si_trainer.train(train, val, epochs=3, batch_size=256)
    si_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        loader = DataLoader(
            train_tasks[i + 1],
            batch_size=128,
            shuffle=True,
            generator=torch.Generator().manual_seed(42),
        )
        samples = next(iter(loader))
        si_trainer.compute_rashomon_set(test, si_batch=samples, prune_prop=0.8)

To keep top 20%, found global SI threshold: 0.0014
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.2177 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.70
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.92,  Min acc soft=0.92


100%|█████████████████████████████████████████| 200/200 [00:12<00:00, 15.59it/s, size=167.23, obj=0.027, min_soft_acc=0.758]


Final bbox:  Obj=0.03,  Size=167.23,  Min acc hard=0.75,  Min acc soft=0.72
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['3.39', '23.40', '54.74', '70.77', '86.87', '104.56', '120.53', '134.56', '151.33', '167.23']
Checkpoint certificates: ['0.92', '0.91', '0.81', '0.85', '0.84', '0.71', '0.76', '0.82', '0.77', '0.75']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|███████████████████████████████| 3/3 [00:20<00:00,  6.98s/it, val_loss=1.8731, val_acc=0.1757, proj=2]


Test Results: [(0.6975, 0.9433), (1.8739, 0.1725)]
To keep top 20%, found global SI threshold: 0.0011
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.0108 (Positive = violated)
Computing Rashomon set within outer box of size: 54.74
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.15
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.16,  Min acc soft=0.16


...

Test Results: [(0.6923, 0.9433), (1.9251, 0.1725), (9.0467, 0.0)]
To keep top 20%, found global SI threshold: 0.0013
---------------------------- Computing Rashomon set ----------------------------


Training Epochs: 100%|██████████████████████████████| 3/3 [00:19<00:00,  6.34s/it, val_loss=10.0810, val_acc=0.0000, proj=0]


Test Results: [(0.6922, 0.9433), (1.9251, 0.1725), (9.1525, 0.0), (10.1577, 0.0)]


Training Epochs: 100%|███████████████████████████████| 3/3 [00:23<00:00,  7.72s/it, val_loss=8.3027, val_acc=0.0000, proj=0]


Test Results: [(0.6923, 0.9433), (1.9252, 0.1725), (9.1609, 0.0), (10.2656, 0.0), (8.4119, 0.0)]


### Task Incremental Learning

In [17]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [18]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|████████████████████████████████████████| 3/3 [00:20<00:00,  6.67s/it, val_loss=0.0369, val_acc=0.991]

Test Results: [(0.0209, 0.993)]


[(0.0209, 0.993)]

In [ ]:
si_trainer = SITrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.7,
    primal_learning_rate=0.5,
    paradigm="TIL",
)

loader = DataLoader(
    train_tasks[1],
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
samples = next(iter(loader))
# Compute bounds for task 0
si_trainer.compute_rashomon_set(
    test_tasks[0], si_batch=samples, prune_prop=0.8, context_id=0, si_context_id=1
)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    si_trainer.train(train, val, epochs=3, batch_size=256, context_id=i)
    si_trainer.test(test_tasks[0 : i + 1], context_list=list(range(i + 1)))
    if i < len(train_tasks) - 2:
        loader = DataLoader(
            train_tasks[i + 1],
            batch_size=128,
            shuffle=True,
            generator=torch.Generator().manual_seed(42),
        )
        samples = next(iter(loader))
        si_trainer.compute_rashomon_set(
            test, si_batch=samples, prune_prop=0.8, context_id=i, si_context_id=i + 1
        )

To keep top 20%, found global SI threshold: 0.0003
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.2883 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.70
Initial bbox:  Obj=14.94,  Size=92320.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|██████████████████████████████████████| 200/200 [00:11<00:00, 17.68it/s, size=92322.30, obj=14.941, min_soft_acc=0.926]


Final bbox:  Obj=14.94,  Size=92322.30,  Min acc hard=0.93,  Min acc soft=0.93
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['92321.39', '92321.40', '92321.27', '92320.64', '92320.72', '92321.59', '92322.60', '92321.09', '92321.30', '92322.30']
Checkpoint certificates: ['0.79', '0.92', '0.96', '0.99', '0.99', '0.95', '0.75', '0.99', '0.99', '0.93']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|███████████████████████████████| 3/3 [00:18<00:00,  6.24s/it, val_loss=0.1282, val_acc=0.9655, proj=6]


Test Results: [(0.0202, 0.9935), (0.1124, 0.9667)]
To keep top 20%, found global SI threshold: 0.0002
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.2547 (Positive = violated)
Computing Rashomon set within outer box of size: 92322.60
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.70
Initial bbox:  Obj=14.94,  Size=92320.00,  Min acc hard=0.95,  Min acc soft=0.95


...

Final bbox:  Obj=7.47,  Size=46160.73,  Min acc hard=0.97,  Min acc soft=0.97
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['46160.73', '46160.73', '46160.73', '46160.73', '46160.73', '46160.73', '46160.73', '46160.73', '46160.73', '46160.73']
Checkpoint certificates: ['0.97', '0.97', '0.97', '0.97', '0.97', '0.97', '0.97', '0.97', '0.97', '0.97']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|███████████████████████████████| 3/3 [00:18<00:00,  6.08s/it, val_loss=0.0994, val_acc=0.9674, proj=0]


Test Results: [(0.0202, 0.9935), (0.1124, 0.9672), (0.0267, 0.993), (0.0718, 0.9765)]


Training Epochs: 100%|███████████████████████████████| 3/3 [00:24<00:00,  8.04s/it, val_loss=0.0652, val_acc=0.9828, proj=0]


Test Results: [(0.0202, 0.9935), (0.1124, 0.9672), (0.0267, 0.993), (0.0718, 0.9765), (0.0666, 0.9912)]


### Domain Incremental Learning

In [20]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model(device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [21]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [22]:
trainer = SimpleTrainer(model, paradigm="DIL", domain_map_fn=domain_map_fn)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|████████████████████████████████████████| 3/3 [00:21<00:00,  7.20s/it, val_loss=0.0352, val_acc=0.991]


Test Results: [(0.0208, 0.994)]


[(0.0208, 0.994)]

In [ ]:
si_trainer = SITrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.7,
    primal_learning_rate=0.5,
    paradigm="DIL",
    domain_map_fn=domain_map_fn,
)

loader = DataLoader(
    train_tasks[1],
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
samples = next(iter(loader))
# Compute bounds for task 0
si_trainer.compute_rashomon_set(test_tasks[0], si_batch=samples, prune_prop=0.8)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    si_trainer.train(train, val, epochs=3, batch_size=256)
    si_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        loader = DataLoader(
            train_tasks[i + 1],
            batch_size=128,
            shuffle=True,
            generator=torch.Generator().manual_seed(42),
        )
        samples = next(iter(loader))
        si_trainer.compute_rashomon_set(test, si_batch=samples, prune_prop=0.8)

To keep top 20%, found global SI threshold: 0.0003
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.2883 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.70
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|██████████████████████████████████████████| 200/200 [00:10<00:00, 18.57it/s, size=54.06, obj=0.009, min_soft_acc=0.702]


Final bbox:  Obj=0.01,  Size=54.06,  Min acc hard=0.73,  Min acc soft=0.73
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['7.25', '17.91', '23.07', '34.11', '39.10', '42.88', '46.18', '49.78', '51.57', '54.06']
Checkpoint certificates: ['0.73', '0.80', '0.88', '0.70', '0.72', '0.72', '0.73', '0.73', '0.73', '0.73']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|███████████████████████████████| 3/3 [00:18<00:00,  6.19s/it, val_loss=2.8047, val_acc=0.1881, proj=0]


Test Results: [(0.0241, 0.994), (2.9254, 0.1795)]
To keep top 20%, found global SI threshold: 0.0001
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.0555 (Positive = violated)
Computing Rashomon set within outer box of size: 7.25
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.17
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.23,  Min acc soft=0.23


...

Test Results: [(0.0239, 0.994), (2.9342, 0.1805), (0.4218, 0.8223)]
To keep top 20%, found global SI threshold: 0.0004
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1222 (Positive = violated)
Computing Rashomon set within outer box of size: 0.91


Training Epochs: 100%|███████████████████████████████| 3/3 [00:17<00:00,  5.73s/it, val_loss=3.7161, val_acc=0.3887, proj=0]


Test Results: [(0.0252, 0.9935), (2.9861, 0.1866), (0.4536, 0.8137), (3.4899, 0.4012)]


Training Epochs: 100%|███████████████████████████████| 3/3 [00:22<00:00,  7.54s/it, val_loss=2.4871, val_acc=0.5867, proj=0]


Test Results: [(0.0252, 0.9935), (2.9861, 0.1866), (0.4536, 0.8137), (3.4899, 0.4012), (2.8997, 0.5621)]


### Experiments

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

model = models.get_mnist_model(device="cuda")

trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=1, batch_size=300)
trainer.test(test_tasks[0:2])

In [ ]:
save_model = copy.deepcopy(trainer.model)

In [ ]:
interval_trainer = IntervalTrainer(
    save_model,
    checkpoint=200,
    n_iters=200,
    min_acc_limit=0.8,
    min_acc_increment=0,
    primal_learning_rate=0.5,
    paradigm="CIL",
    seed=0,
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

# Train task 1 until plateau
interval_trainer.train(
    train_tasks[1],
    val_tasks[1],
    batch_size=256,
    epochs=10,
    early_stopping=True,
    patience=5,
    lr=0.01,
)
interval_trainer.test(test_tasks[0:2])

In [ ]:
si_trainer = SITrainer(
    save_model,
    checkpoint=200,
    n_iters=200,
    min_acc_limit=0.8,
    min_acc_increment=0,
    primal_learning_rate=0.5,
    paradigm="CIL",
    seed=0,
)

# Compute bounds for task 0
si_trainer.compute_rashomon_set(test_tasks[0], prune_prop=0)

# Train task 1 until plateau
si_trainer.test(test_tasks[0:2])
si_trainer.train(train_tasks[1], val_tasks[1], batch_size=256, epochs=3)
si_trainer.test(test_tasks[0:2])

In [ ]:
si_trainer = SITrainer(
    save_model,
    checkpoint=200,
    n_iters=200,
    min_acc_limit=0.8,
    min_acc_increment=0,
    primal_learning_rate=0.5,
    paradigm="CIL",
    seed=0,
)

loader = DataLoader(
    train_tasks[1],
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(0),
)
samples = next(iter(loader))
si_trainer.compute_rashomon_set(test_tasks[0], prune_prop=0.8, si_batch=samples)
si_trainer.test(test_tasks[0:2])
si_trainer.train(train_tasks[1], val_tasks[1], epochs=5, batch_size=256)
si_trainer.test(test_tasks[0:2])

In [ ]:
def run_experiment(
    seed, lookahead_steps=10, require_prev=True, pruning_prop=0.7, si_batch_size=128
):
    train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=seed)

    model = models.get_mnist_model(seed=seed, device="cuda")

    train_tasks, _ = zip(*[create_holdout_set(dataset) for dataset in train_tasks])
    trainer = SimpleTrainer(model, seed=seed)
    trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
    trainer.test(test_tasks[0:1])

    save_model = copy.deepcopy(trainer.model)

    interval_trainer = IntervalTrainer(
        save_model,
        checkpoint=200,
        n_iters=200,
        min_acc_limit=0.8,
        min_acc_increment=0,
        primal_learning_rate=0.5,
        paradigm="CIL",
        seed=seed,
    )

    # Compute bounds for task 0
    prev = []
    if require_prev:
        interval_trainer.compute_rashomon_set(test_tasks[0])

        # Train task 1 until plateau
        interval_trainer.train(train_tasks[1], val_tasks[1], batch_size=256, epochs=3)
        print("No pruning performance: ", end="")
        prev = interval_trainer.test(test_tasks[0:2])

    si_trainer = SITrainer(
        save_model,
        checkpoint=200,
        n_iters=200,
        min_acc_limit=0.8,
        min_acc_increment=0,
        primal_learning_rate=0.5,
        paradigm="CIL",
        seed=seed,
    )

    loader = DataLoader(
        train_tasks[1],
        batch_size=si_batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    samples = next(iter(loader))
    si_trainer.compute_rashomon_set(
        test_tasks[0], prune_prop=pruning_prop, si_batch=samples
    )
    si_trainer.train(train_tasks[1], val_tasks[1], epochs=3, batch_size=256)
    new = si_trainer.test(test_tasks[0:2])

    return prev, new

In [ ]:
prev_perf = []
new_perf = []
# reds = []
for i in range(10):
    result = run_experiment(i, lookahead_steps=50, pruning_prop=0.8)
    prev_perf.append(result[0])
    new_perf.append(result[1])
    # reds.append(result[2])

In [ ]:
prev_acc = np.array([val[1][1] for val in prev_perf]).mean()
prev_std = np.array([val[1][1] for val in prev_perf]).std()
new_acc = np.array([val[1][1] for val in new_perf]).mean()
new_std = np.array([val[1][1] for val in new_perf]).std()

# 2. Create a prettier plot
# Use a more balanced figure size
fig, ax = plt.subplots(figsize=(7, 6))

# Define some nice colors
colors = ["#4682B4", "#5FBA7D"]  # Steel Blue and a nice Green

# Plot the bars with improved styling
bars = ax.bar(
    x=["Previous", "New"],
    height=[prev_acc, new_acc],
    yerr=[prev_std, new_std],
    color=colors,
    alpha=0.8,  # Make bars slightly transparent
    edgecolor="black",  # Add a crisp black edge
    capsize=10,  # THIS IS KEY: Adds caps to the error bars
    ecolor="black",  # Color of the error bar lines
    linewidth=1.5,
)

# 3. Add Labels, Title, and Grid for context
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Comparison of Previous vs. New Performance", fontsize=16, pad=20)
ax.set_xticks(ticks=[0, 1], labels=["No lookahead", "50 step lookahead"], fontsize=12)

# Add a subtle horizontal grid to make comparisons easier
ax.yaxis.grid(True, linestyle="--", which="major", color="grey", alpha=0.3)

# Remove the top and right spines for a cleaner look
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Set a dynamic Y-axis limit for better spacing
ax.set_ylim(0, max(new_acc, prev_acc) + max(new_std, prev_std) + 0.05)

# 4. Add data labels on top of the bars
# This shows the exact mean value on the plot
ax.bar_label(bars, fmt="{:.3f}", padding=5, fontsize=11, color="black")

# Ensure everything fits without overlapping
plt.tight_layout()
plt.show()

In [ ]:
data = defaultdict(list)
# pruning_prop = [0, 0.05, 0.1, 0.3, 0.5, 0.7, 0.8, 0.85]
batch_sizes = [1, 32, 64, 128, 258]
for prop in tqdm(batch_sizes):
    for i in range(3):
        data[prop].append(
            run_experiment(i, pruning_prop=0.8, lookahead_steps=20, si_batch_size=prop)
        )

In [ ]:
x_vals, y_means, y_stds = [], [], []
prev_means = []
prop_mean = []
for key, item in data.items():
    x_vals.append(key)
    # Create the numpy array once
    values = np.array([val[1][1][1] for val in item])
    y_means.append(values.mean())
    y_stds.append(values.std())
    prop_mean.append(np.array([val[0] for val in item]).mean())
    prev_means.append(np.array([val[0][1][1] for val in item]).mean())

# Convert lists to numpy arrays for easier calculations
x_vals = np.array(x_vals)
y_means = np.array(y_means)
y_stds = np.array(y_stds)

# 2. Create a prettier plot
# Use a pre-defined style for a professional look
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(9, 6))

# Define a color for the plot
primary_color, secondary_color = ["#4682B4", "#5FBA7D"]

# 3. Plot the main line with markers
ax.plot(
    x_vals,
    y_means,
    color=primary_color,
    linewidth=2.5,
    marker="o",  # Add circles at each data point
    markersize=8,
    label="Mean Performance",
)

ax.plot(
    x_vals,
    prev_means,
    color=secondary_color,
    linewidth=2.5,
    markersize=8,
    linestyle="--",
    label="Mean Baseline Performance",
)

# 4. Add the shaded error band (the key improvement!)
# The alpha value makes the shade transparent
ax.fill_between(
    x_vals,
    y_means - y_stds,  # Lower bound of the error
    y_means + y_stds,  # Upper bound of the error
    color=primary_color,
    alpha=0.2,
    label="Standard Deviation",
)

# 5. Add Labels, Title, and Legend
ax.set_title("Performance vs. Pruning Level", fontsize=16, pad=20)
ax.set_xlabel("Pruning Proportion", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.legend(loc="upper right", fontsize=11)

# Customize tick parameters
ax.tick_params(axis="both", which="major", labelsize=11)

# plt.plot(x_vals, prop_mean)

# Ensure everything fits nicely
plt.tight_layout()
plt.show()


In [ ]:
data2 = defaultdict(list)
lookaheads = [0, 1, 5, 10, 20, 50, 100, 200]
for lookahead in tqdm(lookaheads):
    for i in range(5):
        data2[lookahead].append(
            run_experiment(
                i, pruning_prop=0.8, require_prev=False, lookahead_steps=lookahead
            )
        )

In [ ]:
x_vals, y_means, y_stds = [], [], []
prop_mean = []
for key, item in data2.items():
    x_vals.append(key)
    # Create the numpy array once
    values = np.array([val[1][1][1] for val in item])
    y_means.append(values.mean())
    y_stds.append(values.std())
    prop_mean.append(np.array([val[0] for val in item]).mean())

# Convert lists to numpy arrays for easier calculations
x_vals = np.array(x_vals)
y_means = np.array(y_means)
y_stds = np.array(y_stds)

# 2. Create a prettier plot
# Use a pre-defined style for a professional look
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(9, 6))

# Define a color for the plot
primary_color = "#007ACC"  # A nice, clear blue

# 3. Plot the main line with markers
ax.plot(
    x_vals,
    y_means,
    color=primary_color,
    linewidth=2.5,
    marker="o",  # Add circles at each data point
    markersize=8,
    label="Mean Performance",
)

# 4. Add the shaded error band (the key improvement!)
# The alpha value makes the shade transparent
ax.fill_between(
    x_vals,
    y_means - y_stds,  # Lower bound of the error
    y_means + y_stds,  # Upper bound of the error
    color=primary_color,
    alpha=0.2,
    label="Standard Deviation",
)

# 5. Add Labels, Title, and Legend
ax.set_title("Performance vs. Lookahead", fontsize=16, pad=20)
ax.set_xlabel("Lookahead steps", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.legend(loc="upper right", fontsize=11)

# Customize tick parameters
ax.tick_params(axis="both", which="major", labelsize=11)

# plt.plot(x_vals, prop_mean)

# Ensure everything fits nicely
plt.tight_layout()
plt.show()


In [ ]:
si_trainer = SITrainer(
    save_model,
    checkpoint=200,
    n_iters=200,
    min_acc_limit=0.8,
    min_acc_increment=0,
    primal_learning_rate=0.5,
    paradigm="CIL",
    seed=0,
)

loader = DataLoader(
    train_tasks[1],
    batch_size=1,
    shuffle=True,
    generator=torch.Generator().manual_seed(0),
)
samples = next(iter(loader))

si_trainer.model = copy.deepcopy(save_model)
si_trainer.compute_rashomon_set(test_tasks[0], prune_prop=0.7, si_batch=samples)
si_trainer.train(train_tasks[1], val_tasks[1], epochs=3, batch_size=256)
si_trainer.test(test_tasks[0:2])

### Hyperparam Tuning

In [ ]:
# HYPERPARAMS
# checkpoint
# n_iters
# primal_learning_rate
# dual_learning_rate
# batch_size
# learning rate
# pruning proportion
# penalty coefficient


def tuning_func(
    checkpoint: int = 100,
    n_iters: int = 300,
    min_acc_limit: float = 0.8,
    min_acc_increment: float = 0.05,
    primal_learning_rate: float = 0.5,
    dual_learning_rate: float = 0.1,
    learning_rate: float = 0.001,
    weight_decay: float = 0.01,
    l2_lambda: float = 0.01,
    unbias_lambda: float = 0.01,
    seed: int = 0,
    initial_epochs: int = 3,
    initial_batch_size: int = 128,
    device="cuda",
):
    SEED = seed
    train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

    model = models.get_mnist_model(seed=SEED, device=device)

    l2 = L2Regulariser(lmbd=l2_lambda)
    unbias = UnbiasRegulariser(lmbd=unbias_lambda)
    regulariser = MultiRegulariser([unbias, l2])

    train_tasks, _ = zip(*[create_holdout_set(dataset) for dataset in train_tasks])
    trainer = SimpleTrainer(model, seed=SEED)
    trainer.train(
        train_tasks[0],
        val_tasks[0],
        epochs=initial_epochs,
        batch_size=initial_batch_size,
        regulariser=regulariser,
    )
    trainer.test(test_tasks[0:1])

    save_model = copy.deepcopy(trainer.model)

    si_trainer = SITrainer(
        save_model,
        checkpoint=checkpoint,
        n_iters=n_iters,
        min_acc_limit=min_acc_limit,
        min_acc_increment=min_acc_increment,
        primal_learning_rate=primal_learning_rate,
        dual_learning_rate=dual_learning_rate,
        paradigm="CIL",
        seed=SEED,
    )

    for i, (train, val, test) in enumerate(
        zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
    ):
        loader = DataLoader(
            train,
            batch_size=128,
            shuffle=True,
            generator=torch.Generator().manual_seed(SEED),
        )

        samples = next(iter(loader))
        si_trainer.compute_rashomon_set(
            test_tasks[i - 1], prune_prop=0.8, si_batch=samples
        )
        assert si_trainer.test(test_tasks[0 : i + 1])[-1][1] == 0, (
            "Prior last task performance needs to be zero."
        )
        si_trainer.train(
            train,
            val,
            epochs=20,
            batch_size=256,
            early_stopping=True,
            patience=10,
            lr=learning_rate,
            weigth_decay=weight_decay,
            regulariser=regulariser,
        )
        results = si_trainer.test(test_tasks[0 : i + 1])

        if not all([res[1] for res in results]):
            print("Catastrophic Forgetting occurred.")
            break

    return results

In [27]:
# It's always a good idea to have the latest version
!pip install wandb --upgrade
!pip install nbformat

import wandb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import copy

# Log in to your W&B account
wandb.login()

True

In [ ]:
sweep_config = {
    "method": "bayes",  # Use Bayesian Optimization
    "metric": {"name": "final_total_accuracy", "goal": "maximize"},
    "parameters": {
        "checkpoint": {"values": [20, 50, 100, 150]},
        "n_iters": {"distribution": "q_uniform", "min": 200, "max": 500, "q": 50},
        "min_acc_limit": {"distribution": "uniform", "min": 0.7, "max": 0.95},
        "min_acc_increment": {"distribution": "uniform", "min": 0.01, "max": 0.1},
        "primal_learning_rate": {
            "distribution": "log_uniform_values",
            "min": 0.01,
            "max": 1.0,
        },
        "dual_learning_rate": {
            "distribution": "log_uniform_values",
            "min": 0.01,
            "max": 1.0,
        },
        "learning_rate": {
            "distribution": "log_uniform_values",
            "min": 0.0001,
            "max": 0.01,
        },
        "weight_decay": {
            "distribution": "log_uniform_values",
            "min": 1.0e-5,
            "max": 1.0e-2,
        },
        "l2_lambda": {
            "distribution": "log_uniform_values",
            "min": 1.0e-4,
            "max": 1.0e-1,
        },
        "unbias_lambda": {
            "distribution": "log_uniform_values",
            "min": 1.0e-4,
            "max": 1.0e-1,
        },
        "seed": {"values": list(range(100))},
        "initial_epochs": {"values": [3, 5, 10, 20, 50]},
        "initial_batch_size": {"values": [32, 64, 128, 256, 512, 1024]},
    },
}

In [ ]:
def train_one_run():
    # This function will be called by the W&B agent for each run.

    # Initialize a W&B run. W&B automatically fills wandb.config
    # with the hyperparameters for this run.
    with wandb.init() as run:
        config = wandb.config

        # Call your main logic with the hyperparameters
        final_results = tuning_func(
            checkpoint=config.checkpoint,
            n_iters=config.n_iters,
            min_acc_limit=config.min_acc_limit,
            min_acc_increment=config.min_acc_increment,
            primal_learning_rate=config.primal_learning_rate,
            dual_learning_rate=config.dual_learning_rate,
            learning_rate=config.learning_rate,
            weight_decay=config.weight_decay,
            l2_lambda=config.l2_lambda,
            unbias_lambda=config.unbias_lambda,
            device="cuda",
            seed=config.seed,
            initial_batch_size=config.initial_batch_size,
            initial_epochs=config.initial_epochs,
        )

        # Process and log the final metrics
        if final_results:
            accuracies = [res[1] for res in final_results]
            avg_accuracy = np.mean(accuracies)

            losses = [res[0] for res in final_results]
            avg_loss = np.mean(losses)

            wandb.log(
                {
                    "final_num_tasks": len(final_results),
                    "final_avg_accuracy": avg_accuracy,
                    "second_task_accuracy": accuracies[1] if len(accuracies) > 1 else 0,
                    "final_avg_loss": avg_loss,
                    "final_total_accuracy": np.sum(accuracies),
                }
            )
        else:
            # Log a failure if catastrophic forgetting happened early
            wandb.log({"final_avg_accuracy": 0.0, "final_total_accuracy": 0.0})

In [30]:
# 1. Initialize the sweep
# Replace "your-project-name" with a name for your W&B project
sweep_id = wandb.sweep(sweep_config, project="class_incremental-sweep")

# 2. Run the agent. This will execute your `train_one_run` function `count` times.
# This cell will run for a long time as it trains the model for each trial.
wandb.agent(sweep_id, function=train_one_run, count=100)

Create sweep with ID: gjnmxnnu
Sweep URL: https://wandb.ai/leo_elm/class_incremental-sweep/sweeps/gjnmxnnu


wandb: Agent Starting Run: 0rsxmd82 with config:
wandb: 	checkpoint: 100
wandb: 	dual_learning_rate: 0.010627799516101898
wandb: 	initial_batch_size: 32
wandb: 	initial_epochs: 3
wandb: 	l2_lambda: 0.0008832239153097163
wandb: 	learning_rate: 0.00028393246701589387
wandb: 	min_acc_increment: 0.015540048589478423
wandb: 	min_acc_limit: 0.9277452631670708
wandb: 	n_iters: 200
wandb: 	primal_learning_rate: 0.03780259647766335
wandb: 	seed: 5
wandb: 	unbias_lambda: 0.06779609726286434
wandb: 	weight_decay: 0.0001590518655653462


wandb: 
wandb: 🚀 View run pious-sweep-12 at: https://wandb.ai/leo_elm/class_incremental-sweep/runs/0q8jkmaz
wandb: Find logs at: wandb/run-20250630_130436-0q8jkmaz/logs


wandb: Ctrl + C detected. Stopping sweep.
